## Hidden Markov Models for Regime Detection
##### An HMM assumes the market exists in some hidden state at any given time (like “bull” or “bear”) that we cannot observe directly. Instead, we observe returns whose statistics depend on that hidden state.

In [3]:
!pip install yfinance hmmlearn scikit-learn seaborn

In [4]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf

from hmmlearn.hmm import GaussianHMM

from scipy.stats import wasserstein_distance

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt

In [ ]:
ticker = "^GSPC"  # S&P500

data = yf.download(
    ticker,
    start="2010-01-01",
    auto_adjust=True
)

close = data["Close"]

returns = np.log(close / close.shift(1)).dropna()

df = pd.DataFrame(index=returns.index)
df["returns"] = returns

In [ ]:
# Rolling features

df["volatility_20"] = (
    df["returns"]
    .rolling(20)
    .std()
)

df["momentum_20"] = (
    close.pct_change(20)
    .reindex(df.index)
)

df["ma_ratio"] = (
    close / close.rolling(50).mean()
).reindex(df.index)

df = df.dropna()

features = [
    "returns",
    "volatility_20",
    "momentum_20",
    "ma_ratio"
]

X = df[features].copy()

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [ ]:
n_regimes = 3

hmm_model = GaussianHMM(
    n_components=n_regimes,
    covariance_type="full",
    n_iter=1000,
    random_state=42
)

hmm_model.fit(X_scaled)

hidden_states = hmm_model.predict(X_scaled)

df["hmm_regime"] = hidden_states

In [ ]:
regime_stats = (
    df.groupby("hmm_regime")["returns"]
    .agg(["mean", "std", "count"])
)

print(regime_stats)

In [ ]:
plt.figure(figsize=(16, 6))

for regime in range(n_regimes):

    regime_data = df[df["hmm_regime"] == regime]

    plt.scatter(
        regime_data.index,
        close.reindex(regime_data.index),
        s=8,
        label=f"Regime {regime}"
    )

plt.plot(close.index, close.values, alpha=0.3)

plt.title("Hidden Markov Model Regimes")
plt.legend()

plt.show()

In [ ]:
window_size = 60

windows = []
window_dates = []

returns_array = df["returns"].values

for i in range(window_size, len(returns_array)):

    window = returns_array[i - window_size:i]

    windows.append(window)

    window_dates.append(df.index[i])

windows = np.array(windows)

In [ ]:
n_windows = len(windows)

distance_matrix = np.zeros((n_windows, n_windows))

for i in range(n_windows):

    for j in range(i + 1, n_windows):

        dist = wasserstein_distance(
            windows[i],
            windows[j]
        )

        distance_matrix[i, j] = dist
        distance_matrix[j, i] = dist

In [ ]:
n_clusters = 3

kmeans = KMeans(
    n_clusters=n_clusters,
    random_state=42,
    n_init=20
)

wasserstein_labels = kmeans.fit_predict(distance_matrix)

wasserstein_df = pd.DataFrame(
    index=window_dates
)

wasserstein_df["regime"] = wasserstein_labels

In [ ]:
plt.figure(figsize=(16, 6))

for regime in range(n_clusters):

    regime_idx = wasserstein_df[
        wasserstein_df["regime"] == regime
    ].index

    plt.scatter(
        regime_idx,
        close.reindex(regime_idx),
        s=8,
        label=f"Cluster {regime}"
    )

plt.plot(close.index, close.values, alpha=0.3)

plt.title("Wasserstein Clustering Regimes")
plt.legend()

plt.show()

In [ ]:
wasserstein_df["returns"] = (
    df["returns"]
    .reindex(wasserstein_df.index)
)

cluster_stats = (
    wasserstein_df
    .groupby("regime")["returns"]
    .agg(["mean", "std", "count"])
)

print(cluster_stats)

In [ ]:
# Walk-forward HMM regime-based strategy (no look-ahead).
#
# Procedure for each day t in the out-of-sample window:
#   1. Fit a fresh HMM on data[:t] (expanding window).
#   2. Predict the regime for the most recent observation.
#   3. Map regimes to expected Sharpe USING ONLY THE TRAINING SLICE.
#   4. Take +1 position if the predicted regime has positive trained Sharpe.
#   5. Apply that position to NEXT day's return (lag of 1) plus transaction costs.

WARMUP_DAYS  = 252 * 3   # 3y warm-up before first prediction
REFIT_EVERY  = 21        # refit monthly to keep runtime sane
COST_PER_TRADE = 0.0005  # 5 bps per position change

signals = pd.Series(0.0, index=df.index)
current_signal = 0

for i in range(WARMUP_DAYS, len(df)):
    if (i - WARMUP_DAYS) % REFIT_EVERY == 0:
        train_X = scaler.fit_transform(X.iloc[:i])
        wf_model = GaussianHMM(
            n_components=n_regimes,
            covariance_type='full',
            n_iter=500,
            random_state=42,
        )
        wf_model.fit(train_X)

        # Build regime -> Sharpe map using ONLY training labels
        train_labels  = wf_model.predict(train_X)
        train_returns = df['returns'].iloc[:i].values
        regime_sharpe = {}
        for r in range(n_regimes):
            mask = train_labels == r
            if mask.sum() > 20 and train_returns[mask].std() > 0:
                regime_sharpe[r] = (
                    train_returns[mask].mean() / train_returns[mask].std()
                )
            else:
                regime_sharpe[r] = -np.inf
        best_train_regime = max(regime_sharpe, key=regime_sharpe.get)

    # Predict regime for today, applied to TOMORROW's return
    today_X = scaler.transform(X.iloc[[i]])
    today_regime = int(wf_model.predict(today_X)[0])
    current_signal = 1 if today_regime == best_train_regime else 0
    signals.iloc[i] = current_signal

df['signal'] = signals.shift(1).fillna(0)  # apply to next day
df['turnover'] = df['signal'].diff().abs().fillna(0)
df['strategy_returns'] = df['signal'] * df['returns'] - df['turnover'] * COST_PER_TRADE

df['cum_market']   = np.exp(df['returns'].cumsum())
df['cum_strategy'] = np.exp(df['strategy_returns'].fillna(0).cumsum())

# Performance stats over the out-of-sample window only
oos = df.iloc[WARMUP_DAYS:]
oos_returns = oos['strategy_returns']
sharpe_oos  = (oos_returns.mean() / oos_returns.std()) * np.sqrt(252) if oos_returns.std() else np.nan
running_max = oos['cum_strategy'].cummax()
drawdown    = oos['cum_strategy'] / running_max - 1
max_dd      = drawdown.min()
turnover_pa = oos['turnover'].sum() / (len(oos) / 252)

print(f'OOS Sharpe (net of {COST_PER_TRADE*1e4:.0f}bps): {sharpe_oos:.2f}')
print(f'Max drawdown (strategy)             : {max_dd:.2%}')
print(f'Turnover per year                   : {turnover_pa:.1f} flips')

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), sharex=True,
                               gridspec_kw={'height_ratios': [3, 1]})
ax1.plot(df.index, df['cum_market'],   label='Buy & Hold', alpha=0.8)
ax1.plot(df.index, df['cum_strategy'], label='Walk-Forward Regime Strategy', lw=2)
ax1.axvline(df.index[WARMUP_DAYS], color='k', ls='--', alpha=0.4, label='OOS start')
ax1.set_ylabel('Growth of $1')
ax1.set_title('Walk-Forward Regime Strategy vs Buy & Hold')
ax1.legend()

ax2.fill_between(drawdown.index, drawdown.values, 0, color='red', alpha=0.3)
ax2.set_ylabel('Drawdown')
ax2.set_xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
latest_hmm_regime = df["hmm_regime"].iloc[-1]

latest_wasserstein_regime = (
    wasserstein_df["regime"]
    .iloc[-1]
)

print(f"Current HMM regime: {latest_hmm_regime}")
print(f"Current Wasserstein regime: {latest_wasserstein_regime}")